# 05 反馈循环：TTRL

**场景**：测试时强化学习（TTRL，Zuo et al. 2025）在**没有标签**的测试题上训练模型：每道题采样很多个
答案，多数投票的结果当作伪标签，和多数一致的答案得到奖励，然后更新模型，再去做下一轮。它是一个
"生成 → 打分 → 更新 → 再生成"的循环。

通常这要两套系统：vLLM 负责采样，另一个训练框架负责更新，每一轮把权重从训练端同步回推理端。
在 pllm 里，采样和训练用的是同一个引擎、同一份模型：负载声明一个 LoRA 和它的训练方法，
宿主机上的 Python 循环负责投票和组织训练数据，下一次 `generate` 时一起送进去。

这篇实现 TTRL 最简单的形式：对和多数答案一致的样本做一步监督更新（等价于奖励取 0/1、不减基线的
策略梯度）。在 GSM8K 的 16 道题上跑几轮，观察多数答案的准确率和样本之间的一致程度。

In [1]:
import os

os.environ["VLLM_LOGGING_LEVEL"] = "ERROR"  # 只显示 vLLM 的报错，省掉启动日志

import json
import math
import re
import time
from collections import Counter
from dataclasses import dataclass

import torch
from torch.nn import functional as F
from transformers import AutoTokenizer
from vllm import SamplingParams

import pllm
from pllm.host import CausalTrainBatch

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

## 第一步：负载，一个 LoRA 加一个训练方法

- `lora` 是整个实例共享的一份状态（`pllm.InstanceState`），字段 `a`、`b` 是低秩因子。`pllm.Field` 的形状
  函数拿到每个目标投影的输入、输出宽度，所以一个声明就为每层的 q、v 投影各生成一对因子。
- `adapter` 把它作为 LoRA 加到每一层的 q、v 投影上：`y = W x + scale · B A x`。`b` 初始化为零，
  所以训练前模型和原来完全一样。
- `learn` 带 `@pllm.train(state="lora")`。它的参数来自宿主机送来的一个批次（`pllm.batch(...)`），以及
  `pllm.causal_model("adapter")`：一个装着当前 LoRA 的完整因果语言模型，调用它得到 logits，梯度只流向 LoRA。
  方法返回一个标量 loss，框架负责反向和 AdamW。

In [2]:
def lora_init(fields: dict[str, torch.Tensor], generator: torch.Generator) -> None:
    for name, value in fields.items():
        if name.startswith("a_"):
            torch.nn.init.kaiming_uniform_(value, a=math.sqrt(5), generator=generator)
        else:
            value.zero_()


@dataclass(frozen=True)
class TTRL(pllm.Workload):
    rank: int = 8
    learning_rate: float = 1e-4

    @property
    def lora(self) -> pllm.InstanceState:
        return pllm.InstanceState(
            {"a": pllm.Field(lambda inputs, outputs: (self.rank, inputs)),
             "b": pllm.Field(lambda inputs, outputs: (outputs, self.rank))},
            optimizer=pllm.AdamW(self.learning_rate), initialize=lora_init)

    @property
    def adapter(self) -> pllm.model.Adapter:
        targets = (pllm.model.Projection("q_proj"), pllm.model.Projection("v_proj"))  # 每一层
        return pllm.model.Adapter("lora", targets, operator="low_rank", scale=2.0)

    @pllm.train(state="lora")
    def learn(self, input_ids: torch.Tensor = pllm.batch("input_ids"), labels: torch.Tensor = pllm.batch("labels"),
              attention_mask: torch.Tensor = pllm.batch("attention_mask"),
              model: pllm.model.CausalModel = pllm.causal_model("adapter")) -> torch.Tensor:
        logits = model(input_ids, attention_mask=attention_mask)[:, :-1]
        targets = labels[:, 1:]
        answer = targets != -100  # 只在答案 token 上算 loss
        return F.cross_entropy(logits[answer].float(), targets[answer])

`lora_init` 里的字段名是 `a_0, b_0, a_1, ...`：一个字段在每个目标投影上各有一份，按模型里的顺序编号。

## 第二步：宿主机上的投票

这部分是普通 Python：从 `\boxed{}` 里取答案，统计多数，挑出和多数一致的样本，拼成一个
`CausalTrainBatch`（prompt 部分的标签设为 -100，只在答案上算 loss）。为了控制一次训练的显存，
每轮只取多数票最集中的 8 道题，各取一个和多数一致的样本。

In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL)
with open("gsm8k_test_16.json") as file:  # GSM8K 测试集的前 16 题
    tests = json.load(file)


def normalize(text: str) -> str | None:
    try:
        return f"{float(text.replace(',', '').replace('$', '').strip()):g}"
    except ValueError:
        return None


def boxed(text: str) -> str | None:
    found = re.findall(r"\\boxed\{([^{}]*)\}", text)
    return normalize(found[-1]) if found else None


gold = [normalize(row["a"]) for row in tests]
prompts = [tokenizer.apply_chat_template(
    [{"role": "user", "content": row["q"] + "\nPut the final answer in \\boxed{}."}],
    tokenize=False, add_generation_prompt=True) for row in tests]


def vote(outputs) -> tuple[float, float, list[tuple[list[int], list[int]]]]:
    # 返回：多数答案的准确率、多数票占比的平均值、用于训练的 (prompt, 答案) token 序列。
    correct, agreement, candidates = 0, 0.0, []
    for prompt, output, answer in zip(prompts, outputs, gold):
        answers = [boxed(sample.text) for sample in output.outputs]
        counts = Counter(a for a in answers if a is not None)
        if not counts:
            continue
        majority, count = counts.most_common(1)[0]
        correct += majority == answer
        agreement += count / len(answers)
        sample = next(s for s, a in zip(output.outputs, answers) if a == majority)
        candidates.append((count, tokenizer(prompt, add_special_tokens=False).input_ids, list(sample.token_ids)))
    # 多数票最集中的 8 道题，各取一个和多数一致的样本。
    candidates.sort(key=lambda item: -item[0])
    return correct / len(prompts), agreement / len(prompts), [(p, a) for _, p, a in candidates[:8]]


def training_batch(sequences: list[tuple[list[int], list[int]]]) -> CausalTrainBatch:
    width = max(len(p) + len(a) for p, a in sequences)
    ids = torch.full((len(sequences), width), tokenizer.pad_token_id)
    labels = torch.full_like(ids, -100)
    mask = torch.zeros_like(ids)
    for row, (p, a) in enumerate(sequences):
        ids[row, :len(p) + len(a)] = torch.tensor(p + a)
        labels[row, len(p):len(p) + len(a)] = torch.tensor(a)
        mask[row, :len(p) + len(a)] = 1
    return CausalTrainBatch(ids.cuda(), labels.cuda(), mask.cuda())

## 第三步：闭环

每轮对 16 道题各采样 8 个答案（128 个并发请求），投票，把上一轮挑出的样本作为 `batch=` 送进这一轮的
`generate`：框架在这次调用开始时用它做一步 AdamW 更新，更新后的 LoRA 立刻被后续 token 使用。
`max_training_rows` 是一次训练最多接受的序列数，要在 `ExecutionOptions` 里事先声明，框架据此预算显存。

In [4]:
# vLLM 只拿 25% 显存，剩下的留给训练时整模型的前向和反向。
config = pllm.ModelConfig(MODEL, max_model_len=640, max_num_seqs=128, max_num_batched_tokens=2048,
                          gpu_memory_utilization=0.25, seed=0,
                          execution=pllm.ExecutionOptions(max_training_rows=8))
sampling = SamplingParams(n=8, temperature=0.8, top_p=0.95, max_tokens=256, seed=0)

with pllm.make_llm(config, workload=TTRL()) as llm:
    batch = None
    for round_index in range(5):
        start = time.perf_counter()
        outputs = llm.generate(prompts, sampling, batch=batch)
        accuracy, agreement, chosen = vote(outputs)
        print(f"第 {round_index} 轮：多数答案准确率 {accuracy:.2f}，多数票占比 {agreement:.2f}，"
              f"挑出 {len(chosen)} 个样本，用时 {time.perf_counter() - start:.0f}s，"
              f"已更新 {llm.diagnostics().update_count} 次")
        batch = training_batch(chosen)

第 0 轮：多数答案准确率 0.31，多数票占比 0.16，挑出 8 个样本，用时 10s，已更新 0 次
第 1 轮：多数答案准确率 0.31，多数票占比 0.18，挑出 8 个样本，用时 46s，已更新 1 次
第 2 轮：多数答案准确率 0.38，多数票占比 0.17，挑出 8 个样本，用时 44s，已更新 2 次
第 3 轮：多数答案准确率 0.38，多数票占比 0.19，挑出 8 个样本，用时 11s，已更新 3 次
第 4 轮：多数答案准确率 0.38，多数票占比 0.20，挑出 8 个样本，用时 45s，已更新 4 次


五轮下来，多数答案的准确率从 0.31 到 0.38，多数票占比从 0.16 到 0.20。16 道题上这只是一两道题的差别，
**不足以说明效果**：这一篇演示的是闭环怎么搭，不是复现 TTRL 的结论。多数票占比偏低，是因为这个 0.5B 模型
经常不按要求把答案写进 `\boxed{}`，没有答案的题按 0 计。

用时那一列也值得看：有几轮明显更慢（约 45 秒对 10 秒），多出来的时间花在训练步的编译上，pllm 遇到
新的批次形状时要重新编译训练图。

## 小结

- 训练方法读宿主机批次（`pllm.batch(...)`）和带 LoRA 的整模型（`pllm.causal_model(...)`），返回一个标量 loss；
- 投票、奖励、挑样本都是宿主机上的普通 Python，结果通过下一次 `generate(batch=...)` 送进引擎；
- 采样和训练共用一个引擎、一份权重，没有权重同步。

想做 GRPO 这类带优势权重的更新，批次里需要逐样本的权重；`CausalTrainBatch` 目前只有
`input_ids`、`labels`、`attention_mask` 三个字段，还表达不了。pllm 也有更完整的宿主反馈接口：`llm.generate_feedback(...)` 管理多轮重试和投票组，内置的
`pllm.workloads.TTRLWorkload` 用它实现了带重试的 TTRL 变体。

下一篇 [06 读模型内部](06_reading_inside.ipynb) 换一个方向：不训练，只读模型内部的残差流和逐层 logits。